# n_mels Ablasyonu — STFT Makalesi Majör Revizyonu

**Ne yapar:** `hop_length = 256` sabit tutularak üç `n_fft` değerinde (512, 1024, 2048)
`n_mels = 64` ile 5 tohumlu eğitim yapar. Mevcut `n_mels = 128` sonuçlarıyla birebir
karşılaştırılabilir olması için mimari, hiperparametreler, tohumlar ve veri bölmesi
orijinal çalışmayla **aynıdır**.

**Test edilen hipotez:** `n_fft = 512` + `n_mels = 128` kombinasyonunda mel filtre bankası
dejeneredir (rank 109/128; 128 filtrenin 64'ü ≤ 2 FFT bin kapsar). `n_mels = 64`'te üç
`n_fft` değerinde de filtre bankası tam rank olur (64/64).

> 512'nin performans açığı `n_mels = 64`'te kapanıyorsa neden **dejenerasyondur**.
> Açık devam ediyorsa neden **frekans çözünürlüğünün kendisidir**.

**Öznitelikler Drive'a yazılmaz** — her konfigürasyon bellekte üretilir, eğitilir, atılır.
Yalnızca `result.json` ve `best_model.keras` dosyaları Drive'a kaydedilir.

**Çalıştırma:** Hücreleri sırayla çalıştırın. Runtime → GPU seçmeniz süreyi kısaltır.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# =============================================================
# AYARLAR
# =============================================================
import os, json, time, gc
import numpy as np
import librosa
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from collections import Counter
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

ROOT       = "/content/drive/MyDrive/GTZAN PAPER"
AUDIO_DIR  = os.path.join(ROOT, "GTZAN DATA", "genres_original")
MASTER     = os.path.join(ROOT, "extracted_features_with_segments_new",
                          "fft1024_hop256_mel128_seg3")
OUT_ROOT   = os.path.join(ROOT, "ablation_results")
os.makedirs(OUT_ROOT, exist_ok=True)

SR = 22050; TRACK_DURATION = 30; SEGMENT_DURATION = 3
N_MELS  = 64
HOP     = 256
N_FFTS  = [512, 1024, 2048]
SEEDS   = [42, 123, 456, 789, 1024]
BATCH_SIZE = 32
EPOCHS     = 120          # orijinal kodla ayni

print("audio :", os.path.exists(AUDIO_DIR))
print("master:", os.path.exists(MASTER))
print("tf    :", tf.__version__, "| librosa:", librosa.__version__)
print("GPU   :", tf.config.list_physical_devices("GPU"))

audio : True
master: True
tf    : 2.20.0 | librosa: 0.11.0
GPU   : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
# =============================================================
# ORIJINAL ON-UC FONKSIYONLARI (Feature_Extraction.ipynb ile birebir ayni)
# =============================================================
def load_full_audio(file_path, sr=22050, track_duration=30):
    y, _ = librosa.load(file_path, sr=sr, mono=True)
    target_length = int(sr * track_duration)
    if len(y) < target_length:
        y = np.pad(y, (0, target_length - len(y)), mode="constant")
    else:
        y = y[:target_length]
    return y

def split_into_segments(y, sr=22050, segment_duration=3):
    segment_length = int(sr * segment_duration)
    return [y[i*segment_length:(i+1)*segment_length]
            for i in range(len(y) // segment_length)]

def extract_log_mel_spectrogram(y, sr=22050, n_fft=1024, hop_length=256, n_mels=128):
    mel_spec = librosa.feature.melspectrogram(
        y=y, sr=sr, n_fft=n_fft, hop_length=hop_length,
        win_length=n_fft, n_mels=n_mels, power=2.0)
    log_mel = librosa.power_to_db(mel_spec, ref=np.max)
    mean = np.mean(log_mel); std = np.std(log_mel)
    if std < 1e-8: std = 1e-8
    log_mel = (log_mel - mean) / std
    return np.expand_dims(log_mel, axis=-1).astype(np.float32)

In [4]:
# =============================================================
# ANA BOLME VE PARCA SIRASI — orijinal segments.json'dan okunur
# Boylece uretilen X, mevcut train/val/test indeksleriyle birebir hizalanir.
# =============================================================
segs = json.load(open(os.path.join(MASTER, "segments.json")))
y_all = np.load(os.path.join(MASTER, "y.npy"))
SP = os.path.join(MASTER, "track_level_split")
train_idx = np.load(os.path.join(SP, "train_idx.npy"))
val_idx   = np.load(os.path.join(SP, "val_idx.npy"))
test_idx  = np.load(os.path.join(SP, "test_idx.npy"))

seg_rel = ["/".join(s["track_path"].split("/")[-2:]) for s in segs]   # genre/file.wav
tracks  = list(dict.fromkeys(seg_rel))

print(f"segment {len(segs)} | parca {len(tracks)}")
print(f"bolme  train {len(train_idx)} / val {len(val_idx)} / test {len(test_idx)}")
assert len(segs) == len(train_idx) + len(val_idx) + len(test_idx)

# test segmentleri parca basina 10'luk ardisik bloklar halinde
test_track_indices = np.repeat(np.arange(len(test_idx)//10), 10)
y_train = y_all[train_idx]; y_val = y_all[val_idx]; y_test = y_all[test_idx]

segment 9990 | parca 999
bolme  train 6990 / val 1500 / test 1500


In [5]:
# =============================================================
# SESI BIR KEZ BELLEGE AL (~2.6 GB) — 3 konfigurasyon icin tekrar okunmaz
# =============================================================
t0 = time.time()
AUDIO = {}
for i, t in enumerate(tracks):
    AUDIO[t] = load_full_audio(os.path.join(AUDIO_DIR, t), SR, TRACK_DURATION)
    if (i+1) % 200 == 0:
        print(f"  {i+1}/{len(tracks)}  {time.time()-t0:.0f}s")
print(f"ses yuklendi: {len(AUDIO)} parca, {time.time()-t0:.0f}s, "
      f"~{sum(a.nbytes for a in AUDIO.values())/1e9:.2f} GB")

  200/999  126s
  400/999  236s
  600/999  343s
  800/999  451s
ses yuklendi: 999 parca, 554s, ~2.64 GB


In [6]:
# =============================================================
# MODEL — orijinal mimarinin birebir aynisi
# =============================================================
def build_model(input_shape, num_classes=10):
    REG = regularizers.l2(5e-4)
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(32, (3,3), padding="same", activation="relu", kernel_regularizer=REG),
        layers.BatchNormalization(), layers.SpatialDropout2D(0.15), layers.MaxPooling2D((2,2)),
        layers.Conv2D(64, (3,3), padding="same", activation="relu", kernel_regularizer=REG),
        layers.BatchNormalization(), layers.SpatialDropout2D(0.2),  layers.MaxPooling2D((2,2)),
        layers.Conv2D(64, (3,3), padding="same", activation="relu", kernel_regularizer=REG),
        layers.BatchNormalization(), layers.SpatialDropout2D(0.25), layers.MaxPooling2D((2,2)),
        layers.GlobalAveragePooling2D(),
        layers.Dense(64, activation="relu", kernel_regularizer=REG),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation="softmax"),
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
                  loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
                  metrics=["accuracy"])
    return model

def majority_vote(y_true, y_pred, track_indices):
    tt, tp = [], []
    for tid in np.unique(track_indices):
        m = track_indices == tid
        tt.append(y_true[m][0])
        tp.append(Counter(y_pred[m].tolist()).most_common(1)[0][0])
    return np.array(tt), np.array(tp)

def soft_aggregate(probs, track_indices):
    return np.array([int(probs[track_indices == tid].mean(0).argmax())
                     for tid in np.unique(track_indices)])

In [7]:
# =============================================================
# ABLASYON — 3 konfigurasyon x 5 tohum
# =============================================================
all_runs = []

for n_fft in N_FFTS:
    name = f"fft{n_fft}_hop{HOP}_mel{N_MELS}_seg3"
    print("\n" + "="*72); print(f"  {name}"); print("="*72)

    # --- oznitelik cikarimi (bellekte) ---
    t0 = time.time()
    W = 1 + (SR*SEGMENT_DURATION)//HOP
    X = np.zeros((len(segs), N_MELS, W, 1), dtype=np.float32)
    pos = 0
    for t in tracks:
        for seg in split_into_segments(AUDIO[t], SR, SEGMENT_DURATION):
            X[pos] = extract_log_mel_spectrogram(seg, SR, n_fft, HOP, N_MELS)
            pos += 1
    assert pos == len(segs), (pos, len(segs))
    print(f"  oznitelik: {X.shape}  {time.time()-t0:.0f}s")

    X_train, X_val, X_test = X[train_idx], X[val_idx], X[test_idx]
    del X; gc.collect()

    input_shape = X_train.shape[1:]
    y_train_oh = tf.keras.utils.to_categorical(y_train, 10)
    y_val_oh   = tf.keras.utils.to_categorical(y_val, 10)
    y_test_oh  = tf.keras.utils.to_categorical(y_test, 10)

    cfg_dir = os.path.join(OUT_ROOT, name); os.makedirs(cfg_dir, exist_ok=True)
    runs = []

    for run_idx, seed in enumerate(SEEDS):
        seed_dir = os.path.join(cfg_dir, f"seed_{seed}"); os.makedirs(seed_dir, exist_ok=True)
        rpath = os.path.join(seed_dir, "result.json")
        if os.path.exists(rpath):
            print(f"  seed {seed}: zaten var, atlaniyor"); runs.append(json.load(open(rpath))); continue

        print(f"\n  --- seed {seed} ({run_idx+1}/{len(SEEDS)}) ---")
        tf.keras.utils.set_random_seed(seed)
        train_ds = (tf.data.Dataset.from_tensor_slices((X_train, y_train_oh))
                    .shuffle(len(X_train), seed=seed).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
        val_ds  = tf.data.Dataset.from_tensor_slices((X_val,  y_val_oh )).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
        test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test_oh)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

        model = build_model(input_shape)
        callbacks = [
            EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True, verbose=1),
            ModelCheckpoint(os.path.join(seed_dir, "best_model.keras"),
                            monitor="val_loss", save_best_only=True, verbose=0),
            ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6, verbose=1),
        ]
        t0 = time.time()
        hist = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS,
                         callbacks=callbacks, verbose=2)
        train_time = time.time() - t0

        test_loss, seg_acc = model.evaluate(test_ds, verbose=0)
        probs = model.predict(test_ds, verbose=0)
        y_pred_seg = probs.argmax(1)
        yt_track, yp_hard = majority_vote(y_test, y_pred_seg, test_track_indices)
        yp_soft = soft_aggregate(probs, test_track_indices)
        be = int(np.argmin(hist.history["val_loss"]))

        res = {
            "config": name, "seed": seed, "run_idx": run_idx,
            "fft": n_fft, "hop": HOP, "n_mels": N_MELS,
            "input_shape": list(input_shape), "time_frames": int(input_shape[1]),
            "total_params": int(model.count_params()),
            "best_epoch": be + 1, "total_epochs": len(hist.history["loss"]),
            "train_time_sec": round(train_time, 1),
            "train_acc": float(hist.history["accuracy"][be]),
            "val_acc": float(hist.history["val_accuracy"][be]),
            "overfit_gap": float(hist.history["accuracy"][be] - hist.history["val_accuracy"][be]),
            "seg_test_acc": float(seg_acc), "seg_test_loss": float(test_loss),
            "seg_f1_macro": float(f1_score(y_test, y_pred_seg, average="macro")),
            "seg_f1_per_class": f1_score(y_test, y_pred_seg, average=None).tolist(),
            "seg_confusion_matrix": confusion_matrix(y_test, y_pred_seg).tolist(),
            "track_test_acc": float(accuracy_score(yt_track, yp_hard)),
            "track_f1_macro": float(f1_score(yt_track, yp_hard, average="macro")),
            "track_f1_per_class": f1_score(yt_track, yp_hard, average=None).tolist(),
            "track_confusion_matrix": confusion_matrix(yt_track, yp_hard).tolist(),
            "track_test_acc_soft": float(accuracy_score(yt_track, yp_soft)),
            "track_f1_macro_soft": float(f1_score(yt_track, yp_soft, average="macro")),
            "history": {k: [float(x) for x in v] for k, v in hist.history.items()
                        if k in ("accuracy","val_accuracy","loss","val_loss")},
        }
        json.dump(res, open(rpath, "w"), indent=2)
        np.save(os.path.join(seed_dir, "test_probs.npy"), probs.astype(np.float32))
        runs.append(res)
        print(f"  >> seg {seg_acc:.4f} | parca(sert) {res['track_test_acc']:.4f} "
              f"| parca(yumusak) {res['track_test_acc_soft']:.4f} | {train_time:.0f}s")
        del model, probs; tf.keras.backend.clear_session(); gc.collect()

    json.dump(runs, open(os.path.join(cfg_dir, "all_seeds_results.json"), "w"), indent=2)
    all_runs += runs
    del X_train, X_val, X_test; gc.collect()

json.dump(all_runs, open(os.path.join(OUT_ROOT, "ablation_all_results.json"), "w"), indent=2)
print("\nTAMAMLANDI ->", os.path.join(OUT_ROOT, "ablation_all_results.json"))


  fft512_hop256_mel64_seg3
  oznitelik: (9990, 64, 259, 1)  63s

  --- seed 42 (1/5) ---
Epoch 1/120
219/219 - 23s - 107ms/step - accuracy: 0.2084 - loss: 2.3097 - val_accuracy: 0.1087 - val_loss: 2.5620 - learning_rate: 3.0000e-04
Epoch 2/120
219/219 - 5s - 23ms/step - accuracy: 0.3016 - loss: 2.0810 - val_accuracy: 0.2133 - val_loss: 2.2387 - learning_rate: 3.0000e-04
Epoch 3/120
219/219 - 4s - 20ms/step - accuracy: 0.3488 - loss: 1.9777 - val_accuracy: 0.3900 - val_loss: 1.8807 - learning_rate: 3.0000e-04
Epoch 4/120
219/219 - 4s - 20ms/step - accuracy: 0.3730 - loss: 1.9256 - val_accuracy: 0.4240 - val_loss: 1.7973 - learning_rate: 3.0000e-04
Epoch 5/120
219/219 - 5s - 22ms/step - accuracy: 0.4070 - loss: 1.8603 - val_accuracy: 0.4467 - val_loss: 1.7771 - learning_rate: 3.0000e-04
Epoch 6/120
219/219 - 5s - 21ms/step - accuracy: 0.4193 - loss: 1.8305 - val_accuracy: 0.4913 - val_loss: 1.7132 - learning_rate: 3.0000e-04
Epoch 7/120
219/219 - 4s - 20ms/step - accuracy: 0.4436 - loss

In [8]:
# =============================================================
# KARSILASTIRMA — n_mels=64 (yeni) vs n_mels=128 (makaleden)
# =============================================================
ref128 = {512:(0.7843,0.8573), 1024:(0.7921,0.8600), 2048:(0.7991,0.8533)}
print("  n_fft   seg64    seg128    fark   |  parca64  parca128   fark   | parca64(yumusak)")
for n_fft in N_FFTS:
    r = [x for x in all_runs if x["fft"] == n_fft]
    if not r: continue
    s64 = np.mean([x["seg_test_acc"] for x in r])
    t64 = np.mean([x["track_test_acc"] for x in r])
    f64 = np.mean([x["track_test_acc_soft"] for x in r])
    s128, t128 = ref128[n_fft]
    print("  %5d   %.4f   %.4f   %+.4f  |  %.4f   %.4f   %+.4f  |  %.4f"
          % (n_fft, s64, s128, s64-s128, t64, t128, t64-t128, f64))

print("\n  512'nin digerlerine gore acigi:")
for lbl, get in [("n_mels=128 (mevcut)", lambda n: ref128[n][0]),
                 ("n_mels=64  (yeni)  ", lambda n: np.mean([x["seg_test_acc"] for x in all_runs if x["fft"]==n]))]:
    try:
        gap = np.mean([get(1024), get(2048)]) - get(512)
        print(f"    {lbl}: {gap*100:+.2f} puan")
    except Exception:
        pass
print("\n  Acik n_mels=64'te belirgin kuculuyorsa neden mel filtre bankasi dejenerasyonudur.")

  n_fft   seg64    seg128    fark   |  parca64  parca128   fark   | parca64(yumusak)
    512   0.7911   0.7843   +0.0068  |  0.8573   0.8573   +0.0000  |  0.8680
   1024   0.7832   0.7921   -0.0089  |  0.8387   0.8600   -0.0213  |  0.8560
   2048   0.7748   0.7991   -0.0243  |  0.8453   0.8533   -0.0080  |  0.8493

  512'nin digerlerine gore acigi:
    n_mels=128 (mevcut): +1.13 puan
    n_mels=64  (yeni)  : -1.21 puan

  Acik n_mels=64'te belirgin kuculuyorsa neden mel filtre bankasi dejenerasyonudur.
